# AI as a Mathematical Collaborator — Missing weather measurements

**UDSM MATH4SDG · Saturday 21 November 2026 · 60-minute workshop**

**Your challenge:** Use generative AI to build **two different models** for estimating missing daily temperatures at **both** of two imaginary weather stations. Compare the models, challenge AI's reasoning, choose a predictor, and test your choice.

The observations are **entirely synthetic**. They are not TMA measurements and cannot support conclusions about the climate in Tanzania. You may use this fictional dataset in your AI prompts. **Never upload restricted industry or personal data to an external AI assistant without authorization.**

Work alone or in pairs. The goal is not to find the fanciest algorithm: it is to **test what AI suggests using mathematical evidence**.


## 1 · Inspect two incomplete time series (about 9 minutes)

Each row of `ai_weather_development.csv` describes **one day**. It contains 270 consecutive days, **1 January–27 September 2025**, with:

| Column | Meaning |
|---|---|
| `date` | Day of observation |
| `station_a_temp_c` | Temperature at station A (°C), sometimes missing |
| `station_b_temp_c` | Temperature at station B (°C), sometimes missing |
| `humidity_pct` | Humidity (%), available each day |
| `wind_speed_m_s` | Wind speed (m/s), available each day |

A missing CSV value appears as `NaN` in pandas. Sometimes A is missing, sometimes B, and sometimes **both**. We want to predict missing temperatures at **either station**.

**Information rule:** For an estimate on day `t`, we may use observed measurements from **that day** (including the other station if available) and all **earlier** days. We may **not** use later measurements or temperatures deliberately hidden for evaluation. This is reconstruction with information available by the end of the day, not unrestricted interpolation between earlier and later measurements.

Open this notebook inside `notebooks/` in the extracted MATH4SDG repository. Keep the CSV files in `data/exercises/`.

### Read the data


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

file_path = "../data/exercises/ai_weather_development.csv"
development = pd.read_csv(file_path)

display(development.head(8))
print("Days:", len(development))
print("Missing temperatures at A:", development["station_a_temp_c"].isna().sum())
print("Missing temperatures at B:", development["station_b_temp_c"].isna().sum())


### Use earlier days for training, later days for validation

First, convert `date` from text into a date and sort chronologically. Then use the **first 210 days** (1 January–29 July) for training, and the **next 60 days** (30 July–27 September) for validation. This is about 78%/22%; it is a practical choice, **not** a universal rule. A larger training period gives more examples, while a larger validation period provides more reliable comparisons.

In the validation period, we have **deliberately hidden some temperatures that were originally observed**, in addition to genuinely missing readings. The separate file `ai_weather_validation_answers.csv` stores only these deliberately hidden values. This lets us measure error where a true answer exists. **We never grade predictions against genuinely unknown measurements.**

A further 60 later days are reserved for a final test. Its hidden temperatures are obfuscated; you will only recover them after choosing a model.


In [ ]:
development["date"] = pd.to_datetime(development["date"])
development = development.sort_values("date").reset_index(drop=True)

training_days = development.iloc[:210].copy()
validation_days = development.iloc[210:].copy()

print("Training days:", len(training_days))
print("Validation days:", len(validation_days))
print("Training period:", training_days["date"].min().date(), "to", training_days["date"].max().date())
print("Validation period:", validation_days["date"].min().date(), "to", validation_days["date"].max().date())


## 2 · Prepare prediction cases and a baseline (about 6 minutes)

To use the **same modelling method for either station**, we organize the data into **one case per station and day**. A `station_code` of `0` means station A; `1` means station B. The value `target_temp_c` is the temperature we want to estimate for that station.

The helper below also creates a few explanatory variables:

- `other_today_c`: the **observed** temperature at the other station on the same day (may be missing).
- `own_last_c` and `other_last_c`: the **last observed** temperature at the respective stations, strictly **before** that day. These may come from several days earlier.
- Humidity, wind and day of year, plus station identity.

**You do not need to study the helper's implementation now.** Notice its use of `shift(1)`: yesterday or earlier may inform today's prediction, but tomorrow cannot. The helper never fills hidden temperatures before calculating these features.


In [ ]:
FEATURES = [
    "station_code", "other_today_c", "own_last_c", "other_last_c",
    "humidity_pct", "wind_speed_m_s", "day_of_year",
]


def make_cases(days):
    days = days.sort_values("date").reset_index(drop=True)
    a = days["station_a_temp_c"]
    b = days["station_b_temp_c"]

    shared = {
        "date": days["date"],
        "humidity_pct": days["humidity_pct"],
        "wind_speed_m_s": days["wind_speed_m_s"],
        "day_of_year": days["date"].dt.dayofyear,
    }
    station_a = pd.DataFrame({
        **shared, "station_code": 0, "target_temp_c": a,
        "other_today_c": b,
        "own_last_c": a.ffill().shift(1),
        "other_last_c": b.ffill().shift(1),
    })
    station_b = pd.DataFrame({
        **shared, "station_code": 1, "target_temp_c": b,
        "other_today_c": a,
        "own_last_c": b.ffill().shift(1),
        "other_last_c": a.ffill().shift(1),
    })
    return pd.concat([station_a, station_b], ignore_index=True).sort_values(
        ["date", "station_code"]
    ).reset_index(drop=True)


examples = make_cases(development)
display(examples.head(6))


### Define a baseline and the MAE score

Our **baseline** first uses the observed temperature at the **other station on the same day**. If that is missing, it uses the most recent earlier observation from the station we are predicting, or then the other station. As a last resort, it uses the average of observed training temperatures.

We evaluate models by **mean absolute error (MAE)**:

$$\operatorname{MAE}=\frac{1}{n}\sum_{i=1}^n |y_i-\widehat y_i|.$$

Smaller is better; the result is in °C. The supplied function **`score` computes this MAE**: it gets predictions from the chosen predictor and compares them with deliberately hidden, known temperatures. The predictor receives only the seven explanatory columns listed in `FEATURES`, **not the hidden answers**. You do not need to understand every line of `score` before using it.


In [ ]:
def baseline_predictor(train_rows, new_rows):
    predictions = new_rows["other_today_c"].copy()
    predictions = predictions.fillna(new_rows["own_last_c"])
    predictions = predictions.fillna(new_rows["other_last_c"])
    return predictions.fillna(train_rows["target_temp_c"].mean()).to_numpy()


def score(predictor, train_rows, evaluation_cases):
    # Compute MAE against deliberately hidden observed temperatures.
    predictions = np.asarray(
        predictor(train_rows, evaluation_cases[FEATURES].copy()), dtype=float
    )
    if predictions.shape != (len(evaluation_cases),):
        raise ValueError("Return one predicted temperature per evaluation case.")
    if not np.isfinite(predictions).all():
        raise ValueError("Predictions contain missing or infinite values.")
    return mean_absolute_error(evaluation_cases["truth_temp_c"], predictions)


train_examples = examples[
    (examples["date"] <= training_days["date"].max())
    & examples["target_temp_c"].notna()
].copy()

answers = pd.read_csv(
    "../data/exercises/ai_weather_validation_answers.csv", parse_dates=["date"]
)
validation_cases = examples.merge(
    answers, on=["date", "station_code"], how="inner", validate="one_to_one"
)

baseline_mae = score(baseline_predictor, train_examples, validation_cases)
print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
print("Observed station-day training examples:", len(train_examples))
print("Number of deliberately hidden validation temperatures:", len(validation_cases))


## 3 · Ask AI for your first model (about 9 minutes)

Open ChatGPT or another AI assistant. You can adapt this prompt, but use the specified function name:

> We have incomplete daily temperature observations at **two** weather stations. Our aim is to predict missing temperatures at either station. Each training example has a station code (`0` or `1`), observed temperature target `target_temp_c`, and these numeric features: `station_code`, `other_today_c`, `own_last_c`, `other_last_c`, `humidity_pct`, `wind_speed_m_s`, `day_of_year`. Some features contain `NaN`. Same-day measurements are allowed only when actually observed; previous temperatures were constructed using past observations only. Suggest a **simple** prediction model in Python/scikit-learn, and write `first_predict_temperature(train_rows, new_rows)`, returning one prediction per row of `new_rows`. The training rows contain `target_temp_c` but the new rows contain **only** the seven feature columns. Fit preprocessing and the model using training rows only. Do not install packages. Briefly explain the model and its treatment of missing inputs.

Paste the AI-generated function in the next cell, run it, and then calculate its validation MAE. If it fails, ask AI to debug the error, but inspect the code before using it.


In [ ]:
# Paste the definition provided by AI here, then run the cell.
# def first_predict_temperature(train_rows, new_rows):
#     ...
#     return predictions


In [ ]:
first_model_mae = score(first_predict_temperature, train_examples, validation_cases)
print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
print(f"First AI-model validation MAE: {first_model_mae:.3f} °C")


**Record your first experiment (edit this Markdown cell):**

- What did you ask AI, and which mathematical model did it propose?
- Did you need to correct the code?
- How does it handle missing `other_today_c` or previous temperatures?
- What was its validation MAE compared with the baseline?

## 4 · Ask AI for a *different* model (about 9 minutes)

Now ask AI for a **different model family**, not merely different settings for your first model. Possibilities include **linear or ridge regression**, **support vector regression (SVR)**, **decision trees**, **random forests**, and **k-nearest-neighbors regression**. You can also propose a simple mathematical rule of your own. These are alternatives, not a prescribed ranking.

> Propose a mathematical model **substantially different from the first**. Explain why this model might be better or worse for two incomplete temperature time series, including occasions when both current temperatures are missing. Write `second_predict_temperature(train_rows, new_rows)` with exactly the same input and output convention as the first function. Handle missing features, avoid future information, and use only existing Python packages.

If you select SVR or nearest neighbors, ask AI about **feature scaling**. If you select a more flexible model, ask about **overfitting**. Evaluate the second model on **the same validation cases**, using exactly the same MAE function.


In [ ]:
# Paste the second AI-generated function here, then run this cell.
# def second_predict_temperature(train_rows, new_rows):
#     ...
#     return predictions


In [ ]:
second_model_mae = score(second_predict_temperature, train_examples, validation_cases)
print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
print(f"First AI-model validation MAE: {first_model_mae:.3f} °C")
print(f"Second AI-model validation MAE: {second_model_mae:.3f} °C")


## 5 · Ask AI to compare and criticize (about 8 minutes)

Only **after both models have been constructed** should you ask AI to compare them. For example:

> Compare the two models we just tested. What mathematical assumptions differ? Which model is more vulnerable to overfitting or missing measurements? Can either accidentally use future information or concealed temperatures? Propose **one specific, testable improvement**, and explain why it might *fail*. Do not claim that a lower validation MAE proves the model is generally better.

Discuss the answer with your partner. You may implement one improvement **if time allows**, using the same validation cases. Otherwise keep the two original models.

**Record a claim you checked independently:** What did AI claim? How did you test it? Did the evidence support it?


In [ ]:
# Optional: paste an improved predictor here if you have time.
# def improved_predict_temperature(train_rows, new_rows):
#     ...
#     return predictions
# improved_model_mae = score(improved_predict_temperature, train_examples, validation_cases)


## 6 · Choose a predictor *before* the final test (about 2 minutes)

Look at the validation MAEs. Choose a predictor from `baseline_predictor`, `first_predict_temperature`, `second_predict_temperature`, or your optional improved predictor. **Make your own choice.** A more complicated method is not necessarily better.

In the next cell, replace `FUNCTION_NAME` by the actual name of your chosen function and remove the leading `#`. **Do not put parentheses after the function name:** you are selecting the function, not calling it.

Write down your decision and your reason **before revealing final-test temperatures**.


In [ ]:
# Remove the # and replace FUNCTION_NAME with your chosen predictor.
# chosen_predict_temperature = FUNCTION_NAME


## 7 · Reveal the independent final test (about 5 minutes)

The new file `ai_weather_final.csv` contains 60 **later** days, incomplete observations at both stations, and columns `masked_a_c` / `masked_b_c` holding *obfuscated* temperatures for some deliberately concealed observations. Other missing temperatures have no known value and cannot be scored.

**Stop here until you have chosen your predictor.** The instructor will reveal a simple inverse transformation. Complete the function `deobfuscation(x)` below with that revealed mathematical expression. Until then, the function raises an error instead of silently producing incorrect scores.

The code evaluates only the deliberately concealed temperatures, and retrains the chosen modelling method using all **actually observed** development temperatures. Neither function receives the concealed validation or final-test answers as features.


In [ ]:
final_days = pd.read_csv(
    "../data/exercises/ai_weather_final.csv", parse_dates=["date"]
)


def deobfuscation(x):
    # Replace this line with the function revealed by the instructor.
    raise NotImplementedError("Deobfuscation function has not yet been revealed")


In [ ]:
# Construct prediction cases using only observed current/past measurements.
all_days = pd.concat(
    [development, final_days[development.columns]], ignore_index=True
)
all_examples = make_cases(all_days)

final_examples = all_examples[
    all_examples["date"] >= final_days["date"].min()
].copy()
all_training_examples = examples[examples["target_temp_c"].notna()].copy()

# Recover only deliberately concealed answers (the instructor provides the function).
a_answers = final_days[["date", "masked_a_c"]].rename(
    columns={"masked_a_c": "masked_temp_c"}
).assign(station_code=0)
b_answers = final_days[["date", "masked_b_c"]].rename(
    columns={"masked_b_c": "masked_temp_c"}
).assign(station_code=1)
final_answers = pd.concat([a_answers, b_answers], ignore_index=True)
final_answers = final_answers.dropna(subset=["masked_temp_c"])
final_answers["truth_temp_c"] = deobfuscation(final_answers["masked_temp_c"])

final_cases = final_examples.merge(
    final_answers[["date", "station_code", "truth_temp_c"]],
    on=["date", "station_code"], how="inner", validate="one_to_one"
)

baseline_final_mae = score(baseline_predictor, all_training_examples, final_cases)
chosen_final_mae = score(chosen_predict_temperature, all_training_examples, final_cases)
print(f"Final-test baseline MAE: {baseline_final_mae:.3f} °C")
print(f"Final-test chosen-model MAE: {chosen_final_mae:.3f} °C")
print("Final-test temperatures evaluated:", len(final_cases))


## 8 · Compare results across the five groups (final 12 minutes)

Share results with the facilitator. Together we will fill in this table. All groups use the **same held-out cases and MAE definition**.

| Group | First model | Second model | Chosen predictor | Final-test MAE | Better than baseline? |
|---|---|---|---|---:|---|
| 1 | — | — | — | — | — |
| 2 | — | — | — | — | — |
| 3 | — | — | — | — | — |
| 4 | — | — | — | — | — |
| 5 | — | — | — | — | — |

**Common baseline final-test MAE:** ____ °C

**Discuss:**

1. Which different model families did AI suggest? Did groups choose different approaches?
2. Was the lowest validation error also the lowest final-test error?
3. What happened when the *other station's same-day temperature* was also missing?
4. Did an AI explanation survive an independent check? What claim did you reject or revise?
5. What could go wrong with real missing observations—for example, a storm that disrupts both stations for several days?

**Scientific caution:** These are made-up data. Evaluation on deliberately concealed observations does not prove that a model will work when real missingness is systematic, and one small workshop does not determine which AI prompting strategy is best. **Our goal is to practice mathematical judgment while using AI.**
